# Ch 13 · Lab 3 — 모델 저장 / 로드

원본: `06-Sonar-Save-Model.py`

학습한 모델을 파일로 저장 → 메모리에서 지움 → 다시 로드 → 같은 정확도가 나오는지 확인.

## Keras 3 변경 — 저장 포맷
- 책: `model.save("my_model.h5")` (HDF5)
- Keras 3 권장: `model.save("my_model.keras")` (zip 기반 새 포맷). `.h5` 도 여전히 동작하지만 신규 코드는 `.keras` 사용.

## 0. 환경

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import Input, Sequential
from keras.layers import Dense

keras.utils.set_random_seed(0)

print("Keras:", keras.__version__)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.models import load_model

## 1. 데이터 + 분할

In [ ]:
DATA = "../../data/sonar.csv"
df = pd.read_csv(DATA, header=None)
X = df.iloc[:, 0:60].to_numpy(dtype="float32")
y_str = df.iloc[:, 60].to_numpy()
y = LabelEncoder().fit_transform(y_str).astype("float32")
print("X:", X.shape, "y:", y.shape, "분포:", np.bincount(y.astype(int)))


X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

## 2. 학습

In [ ]:
def build_model():
    return Sequential([
        Input(shape=(60,)),
        Dense(24, activation="relu"),
        Dense(10, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

keras.utils.set_random_seed(0)
model = build_model()
model.compile(loss="mean_squared_error", optimizer="adam", metrics=["accuracy"])
model.fit(X_tr, y_tr, epochs=130, batch_size=5, verbose=0)
before_acc = model.evaluate(X_te, y_te, verbose=0)[1]
print(f"학습 직후 test acc: {before_acc:.4f}")

## 3. 저장 → 메모리 비우기 → 로드

In [ ]:
SAVE_PATH = "../../outputs/ch13_sonar.keras"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
model.save(SAVE_PATH)
print(f"saved to {SAVE_PATH} ({os.path.getsize(SAVE_PATH):,} bytes)")

del model
loaded = load_model(SAVE_PATH)
print("loaded back:", type(loaded).__name__)

## 4. 같은 정확도 확인

In [ ]:
after_acc = loaded.evaluate(X_te, y_te, verbose=0)[1]
print(f"학습 직후 test acc : {before_acc:.4f}")
print(f"로드 직후 test acc : {after_acc:.4f}")
assert before_acc == after_acc, "저장/로드 과정에서 모델이 달라졌습니다."
print("✅ 일치")